Рассмотрим содержание датасетов для тестирования промптов. В качестве главных языков для тестирования были выбраны:<br>
Самые популярные языки среди модификаций:
* Китайский;
* Английский;<br>

Дополнительные языки с потенциальными проблемами:<br>
* Немецкий - похож на английский, но имеет другую грамматику и структуру;
* Японский и корейский сложные языки с иероглифами.


## Как происходил отбор датасетов

1. **Брался исходный SQLite-файл** с параллельными парами строк.
2. Для каждой языковой пары отдельно задавался свой профиль отбора:
   - `en -> ru`
   - `zh -> ru`
   - `de -> ru`
   - `ko -> ru`
   - `ja -> ru`

3. Для каждой пары применялись **базовые фильтры**:
   - брались только строки нужной языковой пары;
   - оставлялись только пары с `ph_match_line = 1`;
   - оставлялись только пары с `sem_sim >= min_sem_sim` (семантическое сходство основанное на близости эмбедингов), где порог зависел от языка.

4. Затем строки **сортировались по качеству**:
   - сначала по `sem_sim` по убыванию;
   - при равенстве — по `pair_id`.

5. После этого выполнялось **удаление дублей**:
   - дубликатом считалась пара `(src_text, dst_text)`;
   - сохранялось только первое вхождение.

6. Каждая строка относилась к одному из типов по простым эвристикам:
   - `placeholder_only` — только плейсхолдеры без буквенного текста;
   - `placeholder_mixed` — текст с плейсхолдерами;
   - `numeric_mixed` — текст с числами;
   - `caps_or_keyish` — строка в капсе;
   - `punct_heavy` — строка с высокой долей пунктуации;
   - `short_ui` — короткая UI-строка;
   - `medium_ui` — средняя UI-строка;
   - `long_text` — длинный текст.

7. Дальше делалась **стратифицированная выборка по типам**:
   - для каждого языка был заранее задан лимит по типу;
   - из каждого типа брались первые `N` строк после сортировки.

8. Итог сохранялся в отдельный SQLite-датасет:
   - таблица `samples` — сами выбранные строки;
   - таблица `sample_stats` — сколько строк попало в каждый тип.

После создания датасетов в каждом файле создавалась SQL-view translations, где:
sample_id переименовывался в row_id,
- src_text становился source_text,
- dst_text становился reference_text,
- добавлялись src_lang, dst_lang,
- metadata заполнялся пустым JSON-объектом {}.
Чтобы подпричесать под бенчмарк для перевода.


Скачаем датасеты с hf

In [11]:
import sqlite3
from huggingface_hub import hf_hub_download
from pathlib import Path
import pandas as pd

In [10]:
REPO_ID = "Opupupas/stellaris_parallel_pairs"

FILES = [
    "sample_en_ru.sqlite",
    "sample_zh_ru.sqlite",
    "sample_de_ru.sqlite",
    "sample_ko_ru.sqlite",
    "sample_ja_ru.sqlite",
]

OUT_DIR = (Path.cwd() / "datasets_sqlite").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook cwd:", Path.cwd())
print("Download dir :", OUT_DIR)

paths = []

for filename in FILES:
    path = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset",
        local_dir=str(OUT_DIR),
    )
    paths.append(Path(path).resolve())

print("\nDownloaded:")
for p in paths:
    print(p, "exists =", p.exists())

Notebook cwd: /Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5
Download dir : /Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite

Downloaded:
/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_en_ru.sqlite exists = True
/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_zh_ru.sqlite exists = True
/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_de_ru.sqlite exists = True
/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_ko_ru.sqlite exists = True
/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_ja_ru.sqlite exists = True


Теперь посмотрим на их содержание

In [12]:
for path in paths:
    print(f"\n{path}")

    conn = sqlite3.connect(path)

    df = pd.read_sql("SELECT * FROM translations LIMIT 5", conn)
    display(df)

    conn.close()


/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_en_ru.sqlite


,row_id,source_text,reference_text,src_lang,dst_lang,metadata
0,1,Corruption,Коррупция,en,ru,{}
1,2,Africa,Африка,en,ru,{}
2,3,Refugees,Беженцы,en,ru,{}
3,4,Water,Вода,en,ru,{}
4,5,Crisis?,Кризис?,en,ru,{}



/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_zh_ru.sqlite


,row_id,source_text,reference_text,src_lang,dst_lang,metadata
0,1,我不是在这里吗？,Разве я не здесь?,zh,ru,{}
1,2,我们必须保护他们！,Мы должны защитить их!,zh,ru,{}
2,3,你可以加入我们。,Ты можешь присоединиться к нам.,zh,ru,{}
3,4,今天,Сегодня,zh,ru,{}
4,5,这里发生了什么事？,Что здесь случилось?,zh,ru,{}



/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_de_ru.sqlite


,row_id,source_text,reference_text,src_lang,dst_lang,metadata
0,1,Tom,Том,de,ru,{}
1,2,Nein.,Нет.,de,ru,{}
2,3,Afrika,Африка,de,ru,{}
3,4,Wasser,Вода,de,ru,{}
4,5,Nein!,Нет!,de,ru,{}



/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_ko_ru.sqlite


,row_id,source_text,reference_text,src_lang,dst_lang,metadata
0,1,Tom,Том,ko,ru,{}
1,2,톰,Том,ko,ru,{}
2,3,Richard,Ричард,ko,ru,{}
3,4,미얀마,Мьянма,ko,ru,{}
4,5,러시아,Россия,ko,ru,{}



/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_ja_ru.sqlite


,row_id,source_text,reference_text,src_lang,dst_lang,metadata
0,1,民主主義,Демократия,ja,ru,{}
1,2,你是谁？,Кто вы?,ja,ru,{}
2,3,北朝鮮,Северная Корея,ja,ru,{}
3,4,古代の宮殿,Древний дворец,ja,ru,{}
4,5,ブラジリア,Бразилиа,ja,ru,{}


In [17]:
dfs = []
for path in paths:
    conn = sqlite3.connect(path)

    df = pd.read_sql("""
        SELECT sample_type, COUNT(*) as cnt
        FROM samples
        GROUP BY sample_type
    """, conn)

    conn.close()

    df = df.set_index("sample_type")
    df.columns = [path.stem]
    dfs.append(df)


final_df = pd.concat(dfs, axis=1).fillna(0).astype(int)
display(final_df)

,sample_en_ru,sample_zh_ru,sample_de_ru,sample_ko_ru,sample_ja_ru
sample_type,,,,,
caps_or_keyish,12,0,4,0,1
long_text,1200,458,600,400,400
medium_ui,1500,1200,700,500,500
numeric_mixed,500,400,200,150,150
placeholder_mixed,1000,800,400,300,300
placeholder_only,247,1,65,6,0
punct_heavy,7,37,1,36,9
short_ui,1500,1200,700,500,500


Базовые типы набрались почти везде
Полностью или почти полностью набраны:

- `short_ui`
- `medium_ui`
- `numeric_mixed`
- `placeholder_mixed`

`long_text` просел не везде одинаково
- `en`: 1200 — квота набрана
- `de`: 600 — квота набрана
- `ko`: 400 — квота набрана
- `ja`: 400 — квота набрана
- `zh`: 458 — недобор

Но это не страшно для иероглифов

Самый сильный недобор — у редких специальных классов
Особенно это видно у:

- `placeholder_only`
- `punct_heavy`
- `caps_or_keyish`

Примеры:
- `placeholder_only`:
  - `en`: 247 из 300
  - `zh`: 1 из 250
  - `de`: 65 из 120
  - `ko`: 6 из 100
  - `ja`: 0 из 100

- `caps_or_keyish`:
  почти пусто во всех языках, кроме очень небольшого числа строк

- `punct_heavy`:
  тоже почти везде мало

Есть на это несколько причин<br>
Чисто плейсхолдеры на этапе распознавания языков вполне могли отъехать в англ.<br>
На этапе удаления/очистки они были шлепнуты <br>
В целом - не важно, капса и так почти не было, пунктуации у иероглифов можно было не ждать, а чисто заглушки вероятно будут перенесены без перевода - это моя новая идея для облегчения перевода, работаю на статистически основанном детекторе разметки.<br>

Все это нормально и на нашу статистику с тестами не слишком повлияет, если что сделаем поправку при трактовке.


In [14]:
for path in paths:
    print(f"\n{path}")
    conn = sqlite3.connect(path)

    df = pd.read_sql("""
        SELECT
            MIN(sem_sim) as min_sim,
            AVG(sem_sim) as avg_sim,
            MAX(sem_sim) as max_sim
        FROM samples
    """, conn)

    display(df)
    conn.close()

,min_sim,avg_sim,max_sim
0,0.702314,0.947037,1.0


,min_sim,avg_sim,max_sim
0,0.618318,0.923974,0.996812


,min_sim,avg_sim,max_sim
0,0.684272,0.958813,0.998917


,min_sim,avg_sim,max_sim
0,0.571879,0.933366,0.999696


,min_sim,avg_sim,max_sim
0,0.614893,0.921119,0.997501


Здесь все хорошо, качество совпадение с референсом должно быть отличное.

In [15]:
for path in paths:
    print(f"\n{path}")
    conn = sqlite3.connect(path)

    df = pd.read_sql("""
        SELECT
            AVG(LENGTH(src_text)) as avg_len,
            MAX(LENGTH(src_text)) as max_len
        FROM samples
    """, conn)

    display(df)
    conn.close()


/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_en_ru.sqlite


,avg_len,max_len
0,64.341435,2316



/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_zh_ru.sqlite


,avg_len,max_len
0,41.588623,4113



/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_de_ru.sqlite


,avg_len,max_len
0,73.053184,1748



/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_ko_ru.sqlite


,avg_len,max_len
0,50.802326,1684



/Users/sergejpolunin/PycharmProjects/LLM-Translator/checkpoint5/datasets_sqlite/sample_ja_ru.sqlite


,avg_len,max_len
0,41.753226,2958


Здесь тоже никаких проблем.<br>
Так-то на этом краткий обзор тестовых датасетов заканчивается.<br>
Разве что одной важной детали нет - у разных языков пары строк разные <br>